In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader
file_info = {
    "file_path": "data/books_data.csv",
    "file_fields": [
        "Title",
        "description",
        "authors",
        "image",
        "previewLink",
        "publisher",
        "publishedDate",
        "infoLink",
        "categories",
        "ratingsCount",
    ],
    "key": ["Title"],
    "content_column": ["description"]
}
# file_info = {
#     "file_path": "data/books_rating.csv",
#     "file_fields": [
#         "Id",
#         "Title",
#         "Price",
#         "User_id",
#         "profileName",
#         "review_helpfulness",
#         "review_score",
#         "review_time",
#         "review_summary",
#         "review_text"
#     ],
#     "key": ["id"],
#     "content_column": ["review_text"]
# }

loader = CSVLoader(
    file_path=file_info["file_path"],
    csv_args={
        "delimiter": ",",
        "quotechar": '"',
        "fieldnames": file_info["file_fields"]
    },
    encoding="utf-8",
    metadata_columns=file_info["key"],
    content_columns=file_info["content_column"]
)

In [3]:
data = loader.load()
data

[Document(metadata={'source': 'data/books_data.csv', 'row': 0, 'Title': 'Title'}, page_content='description: description'),
 Document(metadata={'source': 'data/books_data.csv', 'row': 1, 'Title': 'Its Only Art If Its Well Hung!'}, page_content='description: '),
 Document(metadata={'source': 'data/books_data.csv', 'row': 2, 'Title': 'Dr. Seuss: American Icon'}, page_content='description: Philip Nel takes a fascinating look into the key aspects of Seuss\'s career - his poetry, politics, art, marketing, and place in the popular imagination." "Nel argues convincingly that Dr. Seuss is one of the most influential poets in America. His nonsense verse, like that of Lewis Carroll and Edward Lear, has changed language itself, giving us new words like "nerd." And Seuss\'s famously loopy artistic style - what Nel terms an "energetic cartoon surrealism" - has been equally important, inspiring artists like filmmaker Tim Burton and illustrator Lane Smith. --from back cover'),
 Document(metadata={'so

In [4]:
data[1].to_json()

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'document', 'Document'],
 'kwargs': {'metadata': {'source': 'data/books_data.csv',
   'row': 1,
   'Title': 'Its Only Art If Its Well Hung!'},
  'page_content': 'description: ',
  'type': 'Document'}}

## Splitting Documents

There are good reasons to split documents. As explained in [LangChain's Documentation](https://python.langchain.com/docs/concepts/text_splitters/#why-split-documents):


- Handling non-uniform document lengths: Real-world document collections often contain texts of varying sizes. Splitting ensures consistent processing across all documents.
- Overcoming model limitations: Many embedding models and language models have maximum input size constraints. Splitting allows us to process documents that would otherwise exceed these limits.
- Improving representation quality: For longer documents, the quality of embeddings or other representations may degrade as they try to capture too much information. Splitting can lead to more focused and accurate representations of each section.
- Enhancing retrieval precision: In information retrieval systems, splitting can improve the granularity of search results, allowing for more precise matching of queries to relevant document sections.
- Optimizing computational resources: Working with smaller chunks of text can be more memory-efficient and allow for better parallelization of processing tasks.

## Text Splitters in LangChain

LangChain contains a family of [document splitters](https://docs.langchain.com/oss/python/integrations/splitters/index):

- Length-based: simple and intuitive approach that ensures a specific text length. Can be based on [characters](https://python.langchain.com/docs/how_to/character_text_splitter/) or [tokens](https://python.langchain.com/docs/how_to/split_by_token/).
- Text structure-based: tries to use the natural structure of text, including paragraphs, sentences, and words. More specifically:

    + The [RecursiveCharacterTextSplitter](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) attempts to keep larger units (e.g., paragraphs) intact.
    + If a unit exceeds the chunk size, it moves to the next level (e.g., sentences).
    + This process continues down to the word level if necessary.

- Document Structure-based: Uses the structure of documents in specific formats, including Markdown, HTML, and JSON.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000, 
    chunk_overlap=200, 
    length_function = len, 
    add_start_index = True
)

In [40]:
chunks = text_splitter.split_documents(data)
print(f'Split {len(data)} reviews (documents) into {len(chunks)} chunks.' )

Split 5001 reviews (documents) into 5138 chunks.


Notice that the output documents (the "chunks") include the keys:

- `seq_num`: a sequential number identifying each of the original documents. 
- `start_index`: the starting index for the chunk.
- `page_content`: text of the document chunk.

In [41]:
chunks

[Document(metadata={'source': 'data/books_data.csv', 'row': 0, 'Title': 'Title', 'start_index': 0}, page_content='description: description'),
 Document(metadata={'source': 'data/books_data.csv', 'row': 1, 'Title': 'Its Only Art If Its Well Hung!', 'start_index': 0}, page_content='description:'),
 Document(metadata={'source': 'data/books_data.csv', 'row': 2, 'Title': 'Dr. Seuss: American Icon', 'start_index': 0}, page_content='description: Philip Nel takes a fascinating look into the key aspects of Seuss\'s career - his poetry, politics, art, marketing, and place in the popular imagination." "Nel argues convincingly that Dr. Seuss is one of the most influential poets in America. His nonsense verse, like that of Lewis Carroll and Edward Lear, has changed language itself, giving us new words like "nerd." And Seuss\'s famously loopy artistic style - what Nel terms an "energetic cartoon surrealism" - has been equally important, inspiring artists like filmmaker Tim Burton and illustrator Lan

## Batch Embeddings

We now have a large number of documents for which we need embeddings. We could use a direct call to the Embeddings API. However, here we demonstrate how to request embeddings using the [Batch API](https://platform.openai.com/docs/api-reference/batch). From the documentation:

The Batch API is used to send asynchronous groups of requests. This API offers lower costs, a separate pool of significantly higher rate limits, and a clear 24-hour turnaround time. The service is ideal for processing jobs that don't require immediate responses. 

A couple of useful references are: 

- [Batch API Guide](https://platform.openai.com/docs/guides/batch)
- [API Reference](https://platform.openai.com/docs/api-reference/batch)

## Creating Batches

The batch process works as follows:

1. Prepare the batch file. Batches start with a .jsonl file where each line contains the details of an individual request to the API.
2. Upload the batch file to input. We must first input the batch file, so that we can reference it below.
3. Create the batch.    
4. Check status of the batch.
5. Retrieve the results.

In addition to the steps above, the API allows us to list all batches and to cancel a batch.

### 1. Prepare the Batch File

Batch processing using the API requires input files to follow a specific format. 

A few notes from the [documentation](https://platform.openai.com/docs/guides/batch#1-prepare-your-batch-file)

+ Batches start with a .jsonl file where each line contains the details of an individual request to the API. 
+ The available endpoints are:

    - Responses API: /v1/responses
    - Chat Completions API: /v1/chat/completions 
    - Embeddings API: /v1/embeddings 
    - Completions API: /v1/completions 
    - Moderations API: /v1/moderations 

+ For a given input file, the parameters in each line's body field are the same as the parameters for the underlying endpoint. 
+ Each request **must include a unique custom_id value**, which you can use to reference results after completion. 

#### Rate Limits

It is important to keep in mind the [API's rate limits](https://platform.openai.com/docs/guides/batch#rate-limits):


+ **Per-batch limits**: A single batch may include up to 50,000 requests, and a batch input file can be up to 200 MB in size. Note that /v1/embeddings batches are also restricted to a maximum of 50,000 embedding inputs across all requests in the batch.
+ **Enqueued prompt tokens per model**: Each model has a maximum number of enqueued prompt tokens allowed for batch processing. You can find these limits on the [Platform Settings](https://platform.openai.com/settings/organization/limits) page.

It is important to note: 

> There are no limits for output tokens or number of submitted requests for the Batch API today. Because Batch API rate limits are a new, separate pool, using the Batch API will not consume tokens from your standard per-model rate limits, thereby offering you a convenient way to increase the number of requests and processed tokens you can use when querying our API 



We must create files that contain the `page_content` and an identifier that would arguably include important metadata (like 'reviewid' and a chunk identifier) of our document chunks. We also want to create files that are within the rate limits (i.e., at most 50,000 documents per batch).

The batch definition jsonl should contain one line per request. [Each request is defined as](https://cookbook.openai.com/examples/batch_processing#creating-the-batch-file):

```
{
    "custom_id": <REQUEST_ID>,
    "method": "POST",
    "url": "/v1/chat/completions",
    "body": {
        "model": <MODEL>,
        "messages": <MESSAGES>,
        // other parameters
    }
}
```


In [ ]:
chunks[2].page_content


'description: Philip Nel takes a fascinating look into the key aspects of Seuss\'s career - his poetry, politics, art, marketing, and place in the popular imagination." "Nel argues convincingly that Dr. Seuss is one of the most influential poets in America. His nonsense verse, like that of Lewis Carroll and Edward Lear, has changed language itself, giving us new words like "nerd." And Seuss\'s famously loopy artistic style - what Nel terms an "energetic cartoon surrealism" - has been equally important, inspiring artists like filmmaker Tim Burton and illustrator Lane Smith. --from back cover'

In [45]:
chunks[2].metadata['Title']

'Dr. Seuss: American Icon'

In [13]:
import csv
import json

csv_file_path = file_info["file_path"] 
jsonl_file_path = csv_file_path.split('.')[0] + '.jsonl'
# print(jsonl_file_path)


# Open the CSV file
with open(csv_file_path, 'r', encoding='utf-8') as csv_file:
    # Create CSV reader
    csv_reader = csv.DictReader(csv_file)
    
    # Open JSONL file for writing
    with open(jsonl_file_path, 'w', encoding='utf-8') as jsonl_file:
        # Convert each row to JSON and write to file
        for row in csv_reader:
            json_line = json.dumps(row, ensure_ascii=False)
            jsonl_file.write(json_line + '\n')

print(f"Conversion complete: {csv_file_path} → {jsonl_file_path}")

Conversion complete: data/books_data.csv → data/books_data.jsonl


In [59]:
import json
import os


def prep_batch_file_for_embedding(input:list, output_path:str, max_lines_per_file:int=10000):
    total_lines = len(input)
    num_files = (total_lines // max_lines_per_file) + 1
    print(f'Total lines: {total_lines}, Number of files to create: {num_files}')

    for num_file in range(num_files):
        start_index = num_file * max_lines_per_file
        end_index = min(start_index + max_lines_per_file, total_lines)
        output_file = os.path.join(output_path, f"books_data_batch_{num_file+1}.jsonl")
        print(f'Creating file: {output_file} with lines from {start_index} to {end_index-1}')
        create_single_batch_file(input, start_index, end_index, output_file)

def create_single_batch_file(input, start_index, end_index, output_file):
    with open(output_file, 'w') as outfile:
        for line in input[start_index:end_index]:
            custom_id = (
                    str(line.metadata['Title']) + "_" + 
                    str(line.metadata['row']) + "_" + 
                    str(line.metadata['start_index'])
                )
            content = line.page_content
            out_dict = {
                    "custom_id": custom_id, 
                    "method": "POST", 
                    "url": "/v1/embeddings", 
                    "body": {
                        "model": "text-embedding-3-small", 
                        "input": content
                    }
                }
            outfile.write(json.dumps(out_dict) + '\n')
  
      

In [57]:
chunks[2].metadata

{'source': 'data/books_data.csv',
 'row': 2,
 'Title': 'Dr. Seuss: American Icon',
 'start_index': 0}

In [60]:
prep_batch_file_for_embedding(
    input=chunks, 
    output_path='data/'
)

Total lines: 5138, Number of files to create: 1
Creating file: data/books_data_batch_1.jsonl with lines from 0 to 5137


### 2. Upload the Input File

Before running the batch process, we will upload the files to the API. File management has some useful functions.

#### List available files

In [8]:
from openai import OpenAI

client = OpenAI()
files = client.files.list()


In [9]:
files.to_dict()['data']

[{'id': 'file-Mj1MK57hZNHu2XyBQ7Bp5p',
  'bytes': 815,
  'created_at': 1762734760,
  'filename': 'batch_69112250e2b0819083bc65babb9ba3ce_error.jsonl',
  'object': 'file',
  'purpose': 'batch_output',
  'status': 'processed',
  'expires_at': 1765326760,
  'status_details': None},
 {'id': 'file-Md9Uo7rdzfJrMJC12GL65q',
  'bytes': 211200421,
  'created_at': 1762734757,
  'filename': 'batch_69112250e2b0819083bc65babb9ba3ce_output.jsonl',
  'object': 'file',
  'purpose': 'batch_output',
  'status': 'processed',
  'expires_at': 1765326757,
  'status_details': None},
 {'id': 'file-V7ynEh6ph5LyF3HoWEtQro',
  'bytes': 1499,
  'created_at': 1762734384,
  'filename': 'batch_6911225010dc8190a915442165d0bbcd_error.jsonl',
  'object': 'file',
  'purpose': 'batch_output',
  'status': 'processed',
  'expires_at': 1765326384,
  'status_details': None},
 {'id': 'file-NwLcKe5STVyfxRuLSihQCm',
  'bytes': 211209956,
  'created_at': 1762734381,
  'filename': 'batch_6911225010dc8190a915442165d0bbcd_output.js

#### Remove Files

You can remove files from storage using code like the one below, which deletes all files in the account. 
Note: this is a destructive action that cannot be undone.

In [ ]:
# for file in files.to_dict()['data']:
#     print(f'Deleting file: {file["filename"]}')
#     resp = client.files.delete(file["id"])
#     print(resp)

#### Search and Upload Files

We search for the files that we created and upload them

In [61]:
from glob import glob

batch_files = glob('data/books_data_batch_*.jsonl')
batch_files

['data\\books_data_batch_1.jsonl']

In [62]:
from openai import OpenAI
from tqdm import tqdm
client = OpenAI()


for b_file in tqdm(batch_files):
    batch_input_file = client.files.create(
        file=open(b_file, "rb"), 
        purpose='batch'
    )
    print(batch_input_file)

100%|██████████| 1/1 [00:04<00:00,  4.45s/it]

FileObject(id='file-QGCLwbg8aLaa1VsERZK9YC', bytes=3475974, created_at=1762746753, filename='books_data_batch_1.jsonl', object='file', purpose='batch', status='processed', expires_at=1765338753, status_details=None)


### 3. Create Batches

As before, we can consult the files that we have in store:

In [63]:
# batch_files = client.files.list().to_dict()
# batch_file_ids = [file['id'] for file in batch_files['data']]
# batch_file_ids
batch_file_ids = [batch_input_file.id]
print(batch_file_ids)

['file-QGCLwbg8aLaa1VsERZK9YC']


At a difference with the files API, there is no easy way of removing batches that have a completed or failed state, so the description and status are important. 

Now we can create the batch procedure. For each file, we create the batch with the call below:

In [64]:
my_id = "pabucater_assignment_2"

In [65]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
batch_description = f"Book summary content embeddings ({my_id}) {timestamp}"

for file_id in tqdm(batch_file_ids):
    client.batches.create(
            input_file_id = file_id,
            endpoint="/v1/embeddings",
            completion_window="24h",
            metadata={
                "description": batch_description,
                "timestamp": timestamp
            }
        )

100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


In [66]:
batch_description
# 'Book summary content embeddings (pabucater_assignment_2) 2025-11-09 22:11:32'


'Book summary content embeddings (pabucater_assignment_2) 2025-11-09 22:52:55'

In [91]:
batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
     'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'input_file_id': batch['input_file_id'],  
    'output_file_id': batch['output_file_id']}  
            for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
    ]
batch_info

[{'batch_id': 'batch_69116198464c8190aa3b1463f545bef9',
  'description': 'Book summary content embeddings (pabucater_assignment_2) 2025-11-09 22:52:55',
  'status': 'completed',
  'request_counts': {'completed': 5138, 'failed': 0, 'total': 5138},
  'input_file_id': 'file-QGCLwbg8aLaa1VsERZK9YC',
  'output_file_id': 'file-56E7EFBLQBsDNXuDszTSNC'}]

If you need to cancel a batch, you can use the code below:

In [92]:
# Retrieve details for the failed batch
failed_batch_id = batch_info[0]['batch_id'] 
failed_batch_details = client.batches.retrieve(failed_batch_id)
# Print the details
print(failed_batch_details.to_json())

# for batch in batch_info:
#     client.batches.cancel(batch['batch_id'])


{
  "id": "batch_69116198464c8190aa3b1463f545bef9",
  "completion_window": "24h",
  "created_at": 1762746776,
  "endpoint": "/v1/embeddings",
  "input_file_id": "file-QGCLwbg8aLaa1VsERZK9YC",
  "object": "batch",
  "status": "completed",
  "cancelled_at": null,
  "cancelling_at": null,
  "completed_at": 1762747341,
  "error_file_id": null,
  "errors": null,
  "expired_at": null,
  "expires_at": 1762833176,
  "failed_at": null,
  "finalizing_at": 1762747097,
  "in_progress_at": 1762746778,
  "metadata": {
    "description": "Book summary content embeddings (pabucater_assignment_2) 2025-11-09 22:52:55",
    "timestamp": "2025-11-09 22:52:55"
  },
  "model": "text-embedding-3-small",
  "output_file_id": "file-56E7EFBLQBsDNXuDszTSNC",
  "request_counts": {
    "completed": 5138,
    "failed": 0,
    "total": 5138
  },
  "usage": {
    "input_tokens": 521897,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 0,
    "output_tokens_details": {
      "reasoning

In [93]:
batch_complete = [
    batch  for batch in batch_info if batch['status'] == 'completed'
]
batch_complete

[{'batch_id': 'batch_69116198464c8190aa3b1463f545bef9',
  'description': 'Book summary content embeddings (pabucater_assignment_2) 2025-11-09 22:52:55',
  'status': 'completed',
  'request_counts': {'completed': 5138, 'failed': 0, 'total': 5138},
  'input_file_id': 'file-QGCLwbg8aLaa1VsERZK9YC',
  'output_file_id': 'file-56E7EFBLQBsDNXuDszTSNC'}]

In [ ]:
def get_text_and_embeddings(batch):
    embedding_lines =  get_content_from_file(batch, 'output_file_id')
    text_lines = get_content_from_file(batch, 'input_file_id')
    return embedding_lines, text_lines

def get_content_from_file(batch, key):
    file = client.files.content(batch[key])
    text = file.text
    lines = text.split('\n')
    content_lines = [json.loads(line) for line in lines if line.strip()]
    return content_lines

def create_chroma_inputs(embedding_lines, text_lines):
    chroma_inputs = []
    text_dict = {item['custom_id']: item['body']['input'] for item in text_lines}
    for embed_item in embedding_lines:
        custom_id = embed_item['custom_id']
        text = text_dict.get(custom_id, "")
        chroma_input = {
            'id': embed_item['custom_id'],
            'embedding': embed_item['response']['body']['data'][0]['embedding'],
            'text': text
        }
        chroma_inputs.append(chroma_input)
    return chroma_inputs

def process_batch_for_chromadb(batch):
    embedding_lines, text_lines = get_text_and_embeddings(batch)
    chroma_inputs = create_chroma_inputs(embedding_lines, text_lines)
    return chroma_inputs

def process_batches_for_chromadb(batches):
    all_chroma_inputs = []
    for batch in tqdm(batches, desc="Processing batches"):
        chroma_inputs = process_batch_for_chromadb(batch)
        all_chroma_inputs.extend(chroma_inputs)
    return all_chroma_inputs

In [ ]:
chroma_inputs = process_batches_for_chromadb(batch_complete)

Processing batches: 100%|██████████| 1/1 [00:09<00:00,  9.60s/it]


In [102]:
chroma_inputs[2]

{'id': 'Dr. Seuss: American Icon_2_0',
 'embedding': [0.032323398,
  0.019414289,
  -0.024134973,
  0.021489872,
  0.036752995,
  -0.04819401,
  -0.023932477,
  0.08211207,
  -0.03515834,
  -0.020502703,
  0.020186303,
  -0.028956905,
  -0.04614374,
  0.025160108,
  0.020123024,
  -0.013921589,
  -0.052345175,
  -0.048750874,
  0.0559901,
  0.012542086,
  0.028830346,
  0.019047266,
  0.021578463,
  0.05826818,
  0.007865698,
  0.0042903805,
  0.031690598,
  0.044295967,
  0.048143387,
  -0.033437125,
  -0.026147276,
  -0.022983277,
  -0.027944425,
  0.0030912256,
  -0.014440484,
  0.004008785,
  -0.014883445,
  -0.07239226,
  -0.010143776,
  0.016490756,
  -0.037031427,
  -0.032120902,
  0.030121256,
  0.00937176,
  0.021337999,
  -0.003910701,
  0.023337645,
  -0.015301092,
  0.005268056,
  0.054268885,
  -0.031234983,
  0.010245024,
  0.024286846,
  -0.04009418,
  0.011510623,
  0.018313218,
  -0.034424294,
  -0.006732987,
  -0.032627143,
  -0.02723569,
  0.051130198,
  -0.012080142